# 01. Обзор датасета Real-Life Trial

**Цель:** визуализировать структуру данных, распределение классов и субъектов, а также наглядно показать проблему утечки по идентичности.

**Входные данные:** эмбеддинги MARLIN (`*.npy`) и CSV-файл с разметкой, уже извлечённые на предыдущем этапе.

---

## 0. Установка зависимостей

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn

## 1. Загрузка метаданных

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# ── Пути ────────────────────────────────────────────────────────────────────
# Замените на свои пути к извлечённым эмбеддингам
EMB_BASE = Path("/content/drive/MyDrive/deception_marlin_group_split/embeddings_vit_base")

plt.rcParams.update({
    "figure.dpi": 120,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
COLORS = {"truthful": "#4C72B0", "deceptive": "#DD8452"}

# ── Сбор метаданных из файловой структуры ───────────────────────────────────
rows = []
for label_name, label in [("truthful", 1), ("deceptive", 0)]:
    for npy in sorted((EMB_BASE / label_name).glob("*.npy")):
        subject_id = npy.stem.split("_")[0]
        rows.append({
            "video":          npy.stem,
            "label":          label,
            "label_name":     label_name,
            "subject_id":     subject_id,
            "embedding_path": npy,
        })

df = pd.DataFrame(rows)
print(f"Всего видеозаписей: {len(df)}")
print(df["label_name"].value_counts())
print(f"Уникальных субъектов: {df['subject_id'].nunique()}")

## 2. Распределение классов

Датасет Real-Life Trial почти сбалансирован: 60 правдивых и 59 ложных записей.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))

counts = df["label_name"].value_counts().reindex(["truthful", "deceptive"])
bars = ax.bar(counts.index, counts.values,
              color=[COLORS[c] for c in counts.index],
              width=0.5, edgecolor="white", linewidth=1.2)

for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.5,
            str(v), ha="center", va="bottom", fontsize=13, fontweight="bold")

ax.set_ylim(0, 72)
ax.set_ylabel("Число видеозаписей", fontsize=11)
ax.set_xlabel("Класс", fontsize=11)
ax.set_title("Распределение классов в датасете Real-Life Trial", fontsize=12, pad=10)
ax.set_xticklabels(["Правдивые (truthful)", "Ложные (deceptive)"], fontsize=10)

plt.tight_layout()
plt.savefig("fig_class_distribution.png", bbox_inches="tight")
plt.show()
print("Сохранено: fig_class_distribution.png")

## 3. Распределение клипов по субъектам

Ключевая особенность датасета: **один субъект (s03) представлен 18 клипами класса deceptive** — ~30% всего класса. Это создаёт риск утечки по идентичности при наивном разделении train/test.

In [ ]:
subject_data = (
    df.groupby(["subject_id", "label_name"])
    .size()
    .reset_index(name="n_clips")
)
subject_total = subject_data.groupby("subject_id")["n_clips"].sum()
top_subjects = subject_total.nlargest(15).index.tolist()

sub_pivot = (
    subject_data[subject_data["subject_id"].isin(top_subjects)]
    .pivot(index="subject_id", columns="label_name", values="n_clips")
    .fillna(0)
    .reindex(top_subjects)
)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(top_subjects))
w = 0.38

for i, cls in enumerate(["truthful", "deceptive"]):
    if cls in sub_pivot.columns:
        ax.bar(x + (i - 0.5) * w, sub_pivot[cls],
               width=w, color=COLORS[cls],
               label=cls.capitalize(), alpha=0.92, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(top_subjects, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Число клипов")
ax.set_xlabel("Субъект")
ax.set_title(
    "Распределение видеозаписей по субъектам (топ-15 по суммарному числу клипов)",
    fontsize=11, pad=8
)
ax.legend(fontsize=10)

# Подсветим субъекта s03
if "s03" in top_subjects:
    idx = top_subjects.index("s03")
    ax.axvspan(idx - 0.45, idx + 0.45, color="red", alpha=0.07, zorder=0)
    ax.text(idx, ax.get_ylim()[1] * 0.95, "s03\n(доминирует)",
            ha="center", va="top", fontsize=7.5, color="#bb2200")

plt.tight_layout()
plt.savefig("fig_subject_distribution.png", bbox_inches="tight")
plt.show()
print("Сохранено: fig_subject_distribution.png")

## 4. Проблема утечки по идентичности

Наглядное сравнение: **naive random split** vs **subject-level split (StratifiedGroupKFold)**.

Без контроля идентичности модель «запоминает» субъекта, а не признаки лжи.
Разрыв в 23 п.п. — это не улучшение модели, а методологический артефакт.

In [ ]:
labels     = ["Наивное\nразделение", "По субъектам\n(корректно)"]
values     = [0.736, 0.500]
errors     = [0.05, 0.12]   # приближённые std по фолдам
bar_colors = ["#e07070", "#4C72B0"]

fig, ax = plt.subplots(figsize=(5.5, 4))
bars = ax.bar(labels, values, color=bar_colors, width=0.45,
              yerr=errors, capsize=6, error_kw=dict(elinewidth=1.5, ecolor="#444"))

ax.axhline(0.5, color="grey", linestyle="--", linewidth=1.2, label="Случайное угадывание")

for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02,
            f"{v:.3f}", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.annotate("", xy=(1, 0.736), xytext=(1, 0.500),
            arrowprops=dict(arrowstyle="<->", color="#cc0000", lw=1.5))
ax.text(1.25, 0.618, "Δ = 23.6 п.п.", color="#cc0000", fontsize=9, va="center")

ax.set_ylim(0, 0.88)
ax.set_ylabel("Balanced Accuracy", fontsize=11)
ax.set_title("Утечка по идентичности субъектов:\nвлияние схемы разделения данных", fontsize=11, pad=8)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("fig_leakage_comparison.png", bbox_inches="tight")
plt.show()
print("Сохранено: fig_leakage_comparison.png")

## 5. Сводная статистика датасета

In [ ]:
clips_per_subject = df.groupby("subject_id")["video"].count()

print("=== Сводная статистика Real-Life Trial Dataset ===")
print(f"Всего видеозаписей:          {len(df)}")
print(f"  truthful:                  {(df.label == 1).sum()}")
print(f"  deceptive:                 {(df.label == 0).sum()}")
print(f"Уникальных субъектов:        {df['subject_id'].nunique()}")
print(f"Клипов на субъекта (среднее):{clips_per_subject.mean():.1f}")
print(f"Клипов на субъекта (медиана):{clips_per_subject.median():.1f}")
print(f"Мин / макс клипов:           {clips_per_subject.min()} / {clips_per_subject.max()}")
print(f"Субъект с макс. клипами:     {clips_per_subject.idxmax()} ({clips_per_subject.max()} клипов)")
print(f"Размерность эмбеддинга MARLIN vit_base: 768")

---
**Вывод:** Датасет мал (119 клипов, 54 субъекта) и неравномерен по субъектам. Корректная оценка модели требует разделения *по субъектам*, а не по отдельным клипам.